<a href="https://colab.research.google.com/github/sean838432/TdAI/blob/separate_operational_cycle_models/TdAI_Probabilistic_v2.0_OPERATIONAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
This script trains and validates 10th, 25th, 50th, 75th, and 90th percentile Gradient Boosted Decision Tree (GBDT) models
using a quantile loss function to predict a probabilistic distribution of 21z NBM Td error. It is trained on 6 years of HRRR, NBM, and ASOS data from 2021-2026.


CHANGES FROM v1.0 Probabilistic:
1) Changed the training dataset to 2021-2026 to restrict it to the HRRRv4 operational window
2) Trained separate models for each operational cycle (0300z Day 1, 0300z Day 2, 1500z Day 1, 1500z Day 2)

CHANGES FROM v2.2 Deterministic:
1) Removed NBM Td as a predictor. RH alone should be sufficient. This had minimal effect on performance, especially on the top 25 bust days
2) Added 10th, 25th, 50th, 75th, and 90th TdAI model percentiles and saved as individually trained models


    FEATURE VARIABLES:
        NBM Temperature (C)
        NBM RH (%)
        NBM Sky (%)
        NBM Mixing Height (100s of ft AGL)
        NBM Wind Speed (kts)
        NBM Wind Direction (deg)
        HRRR RH at all levels (%)
        HRRR PWAT
        HRRR 1000mb-850mb Lapse Rate (C/km)
        HRRR 850mb-500mb Lapse Rate (C/km)
        Time of year

    OUTCOME VARIABLE:
        Td error from the 01/13z NBM forecast

    WEIGHTING SCHEME:
        Td error 3-4 F: Weight of 2
        Td error >= 5 F: Weight of 5

    BUST PROPORTIONALITY SCHEME:
        None
'''

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import os
import warnings
import lightgbm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from lightgbm import LGBMRegressor
!pip install metpy
from metpy.plots import SkewT
from metpy.units import units
import math
import joblib
import sklearn

print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)
print("lightgbm:", lightgbm.__version__)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.4/424.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 13.7 MB/s eta 0:00:00
NumPy: 2.0.2
Scikit-learn: 1.6.1
lightgbm: 4.6.0


# **Functions**

In [2]:
def seed_everything(seed=42):
    # 1. Set Python's built-in random seed
    random.seed(seed)
    # 2. Set Numpy's seed (Crucial for your jittering/noise)
    np.random.seed(seed)
    # 3. Set environment variable for any OS-level randomness
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✅ Random state locked at {seed}")

seed_everything(42)


def calculate_lpw_vectorized(df, all_levels):
    """
    Calculates Integrated Water Vapor (LPW) in millimeters (mm) across all
    specified pressure levels simultaneously using high-speed vectorized NumPy operations.
    Designed explicitly for input columns stored natively in Celsius.
    """
    g = 9.80665
    rho_w = 1000.0

    # Pre-allocate specific humidity arrays for each level across all rows
    q_matrix = []

    for lvl in all_levels:
        p = float(lvl)
        dpt_col = f'dpt_{lvl}'  # Ambient temp (t_col) is not mathematically required for LPW depth

        # --- AUGUST-ROCHE-MAGNUS FORMULATION (https://en.wikipedia.org/wiki/Clausius%E2%80%93Clapeyron_relation#Meteorology_and_climatology) ---
        # Takes dpt in Celsius
        e = 6.1094 * np.exp((17.625 * (df[dpt_col])) / (df[dpt_col] + 243.04))

        # Calculate mixing ratio (w) and specific humidity (q) https://glossary.ametsoc.org/wiki/specific-humidity/
        w = 0.622 * e / (p - e)
        q = w / (1.0 + w)

        q_matrix.append(q.values)

    # Shape of matrix: (num_levels, num_rows)
    q_matrix = np.array(q_matrix)

    # Initialize the integration output array
    lpw_total = np.zeros(len(df))

    # Hydrostatic integration layer-by-layer
    for i in range(len(all_levels) - 1):
        p_high = float(all_levels[i])   # Higher pressure (e.g., 1000 hPa)
        p_low = float(all_levels[i+1])  # Lower pressure (e.g., 925 hPa)

        # Pressure differential converted to Pascals (1 hPa = 100 Pa)
        dp = (p_high - p_low) * 100.0

        # Average specific humidity across the bounded layer
        q_avg = (q_matrix[i] + q_matrix[i+1]) / 2.0

        # Hydrostatic integration equation mapping to mm depth
        lpw_total += (q_avg * dp) / (g * rho_w) * 1000.0

    return lpw_total


def calculate_lapse_rate_vectorized(df, p_bottom, p_top):
  """
  Calculates the vertical temperature lapse rate (°C/km) between any two pressure
  levels dynamically using the Hypsometric Equation to determine layer thickness.

  Parameters:
  -----------
  df : pandas.DataFrame
      The pivoted HRRR DataFrame containing columns in 't_LEVEL' format (Celsius).
  p_bottom : int
      The pressure level at the bottom of the layer (e.g., 1000).
  p_top : int
      The pressure level at the top of the layer (e.g., 850).

  Returns:
  --------
  pandas.Series or np.nan
      A vectorized series representing the lapse rate in °C/km.
  """
  t_bottom_col = f't_{p_bottom}'
  t_top_col = f't_{p_top}'

  # 1. Enforce safety validation check for missing levels
  if t_bottom_col not in df.columns or t_top_col not in df.columns:
      return np.nan

  # 2. Extract ambient temperatures at the boundaries (assumed already in Celsius)
  t_bottom = df[t_bottom_col]
  t_top = df[t_top_col]

  # 3. Calculate raw temperature difference (Bottom - Top)
  # Positive = temperature cooling with height
  delta_t = t_bottom - t_top

  # 4. DYNAMIC GEOPOTENTIAL THICKNESS (The Hypsometric Equation)
  # Rd = 287.05 J/(kg*K) [Gas constant for dry air]
  # g  = 9.80665 m/s^2  [Standard gravitational acceleration]
  # Convert mean layer temperature to Kelvin
  t_mean_k = ((t_bottom + t_top) / 2.0) + 273.15

  # Calculate layer thickness in meters, then divide by 1000.0 for kilometers (dz_km)
  dz_meters = (287.05 * t_mean_k / 9.80665) * np.log(float(p_bottom) / float(p_top))
  dz_km = dz_meters / 1000.0

  # 5. Compute the lapse rate
  lapse_rate = delta_t / dz_km

  return lapse_rate

✅ Random state locked at 42


# **Compile Training Dataset**

In [3]:
# --- CONFIGURATION ---
import os
import pandas as pd
import numpy as np

base_path = "/content/drive/MyDrive/Fire Weather Focal Point/TdAI_Project/Development/Data/"
STATIONS = ['CAR']

# 📅 Define dataset date thresholds (YYYY-MM-DD)
START_DATE = '2020-12-02' # To restrict dataset to HRRRv4 only
END_DATE = '2026-12-31'

# Define the 4 target operational cycles mapping inputs and target Day Offsets
CYCLES = [
    {
        'name': '12z_f09',
        'hrrr_file': '12z_f09_Soundings.parquet',
        'nbm_file': '13z.csv',
        'target_day_offset': 0  # Day 1 (Valid Date == Init Date)
    },
    {
        'name': '12z_f33',
        'hrrr_file': '12z_f33_Soundings.parquet',
        'nbm_file': '13z.csv',
        'target_day_offset': 1  # Day 2 (Valid Date == Init Date + 1)
    },
    {
        'name': '00z_f21',
        'hrrr_file': '00z_f21_Soundings.parquet',
        'nbm_file': '01z.csv',
        'target_day_offset': 0  # Day 1 (Valid Date == Init Date)
    },
    {
        'name': '00z_f45',
        'hrrr_file': '00z_f45_Soundings.parquet',
        'nbm_file': '01z.csv',
        'target_day_offset': 1  # Day 2 (Valid Date == Init Date + 1)
    }
]

# Dictionary to collect compiled dataframes separated by cycle
cycle_master_dfs = {cycle['name']: [] for cycle in CYCLES}

print(f"🚀 Starting Master Compilation for {len(STATIONS)} stations across {len(CYCLES)} distinct models...")

for station in STATIONS:
    print(f"\n────────────────── Processing Station: K{station} ──────────────────")

    # --- 1. LOAD ASOS GROUND TRUTH (Only needs to be done once per station) ---
    asos_path = os.path.join(base_path, f"ASOS_Data/K{station}_2020_to_2026_asos.csv")
    if not os.path.exists(asos_path):
        print(f"⚠️ Missing ASOS file for K{station}. Skipping this station entirely.")
        continue

    print(f"   Loading ASOS ground truth...")
    asos_df = pd.read_csv(asos_path)
    asos_df['valid_time'] = pd.to_datetime(asos_df['valid_time']).dt.round('h')
    asos_df = asos_df[['valid_time', 'ASOS Dewpoint (F)']].dropna()

    # --- 2. LOOP THROUGH EACH PREDICTION CYCLE ---
    for cycle in CYCLES:
        c_name = cycle['name']
        print(f"\n   ⚙️ Structuring Cycle: {c_name}...")

        hrrr_path = os.path.join(base_path, f"HRRR_Data/K{station}_{cycle['hrrr_file']}")
        nbm_path = os.path.join(base_path, f"NBM_Data/NBM_Master_Data_K{station}_{cycle['nbm_file']}")

        if not (os.path.exists(hrrr_path) and os.path.exists(nbm_path)):
            print(f"      ⚠️ Missing HRRR or NBM file for {c_name}. Skipping cycle.")
            continue

        try:
            # --- A. LOAD & FILTER NBM BASELINE USING DATE OFFSETS ---
            nbm_df = pd.read_csv(nbm_path)
            nbm_df['valid_time'] = pd.to_datetime(nbm_df['NBM Forecast Valid (UTC)'])
            nbm_df['init_time'] = pd.to_datetime(nbm_df['NBM Initialization Time (UTC)'])

            # Extract just the date components to calculate the Day Offset (0 for Day 1, 1 for Day 2)
            nbm_df['day_offset'] = (nbm_df['valid_time'].dt.normalize() - nbm_df['init_time'].dt.normalize()).dt.days

            # Strictly isolate the correct day's forecast based on our cycle definition
            nbm_df = nbm_df[nbm_df['day_offset'] == cycle['target_day_offset']].copy()

            if nbm_df.empty:
                print(f"      ⚠️ No NBM data found matching Day Offset {cycle['target_day_offset']}. Skipping.")
                continue

            # --- B. LOAD & PIVOT HRRR SOUNDING SAMPLES ---
            hrrr_long = pd.read_parquet(hrrr_path)
            hrrr_long['valid_time'] = pd.to_datetime(hrrr_long['valid_time'])

            # Vectorized pivot mapping 3D profile to 2D features
            hrrr_pivoted = hrrr_long.pivot(
                index='valid_time',
                columns='isobaricInhPa',
                values=['t', 'dpt']
            )

            # Flatten MultiIndex columns
            hrrr_pivoted.columns = [f"{v}_{int(p)}" for v, p in hrrr_pivoted.columns]
            hrrr_pivoted = hrrr_pivoted.reset_index()

            # --- C. METEOROLOGICAL CONVERSIONS ---
            thermal_cols = [c for c in hrrr_pivoted.columns if c.startswith('t_') or c.startswith('dpt_')]
            hrrr_pivoted[thermal_cols] = hrrr_pivoted[thermal_cols] - 273.15

            all_levels = sorted(
                [int(col.split('_')[1]) for col in hrrr_pivoted.columns if col.startswith('t_')],
                reverse=True
            )

            hrrr_pivoted['hrrr_lpw (mm)'] = calculate_lpw_vectorized(hrrr_pivoted, all_levels)

            rh_features_dict = {}
            for lvl in all_levels:
                t_col = f't_{lvl}'
                dpt_col = f'dpt_{lvl}'
                if t_col in hrrr_pivoted.columns and dpt_col in hrrr_pivoted.columns:
                    es = np.exp((17.625 * hrrr_pivoted[t_col]) / (243.04 + hrrr_pivoted[t_col]))
                    e = np.exp((17.625 * hrrr_pivoted[dpt_col]) / (243.04 + hrrr_pivoted[dpt_col]))
                    rh_features_dict[f'rh_{lvl}'] = np.clip(100 * (e / es), 0.0, 100.0)

            new_features_df = pd.DataFrame(rh_features_dict, index=hrrr_pivoted.index)
            hrrr_pivoted = pd.concat([hrrr_pivoted, new_features_df], axis=1)

            hrrr_pivoted['1000mb-700mb Lapse Rate (C/km)'] = calculate_lapse_rate_vectorized(hrrr_pivoted, 1000, 700)
            hrrr_pivoted['700mb-500mb Lapse Rate (C/km)'] = calculate_lapse_rate_vectorized(hrrr_pivoted, 700, 500)

            # --- D. THE NBM-ASOS-HRRR MERGE ---
            station_df = pd.merge(nbm_df, asos_df, on='valid_time', how='inner')
            station_df = pd.merge(station_df, hrrr_pivoted, on='valid_time', how='inner')

            station_df['Target Error (F)'] = station_df['NBM Dewpoint (F)'] - station_df['ASOS Dewpoint (F)']

            # --- E. CALCULATE AND ADD NBM RH ---
            station_df = station_df[pd.to_datetime(station_df['valid_time']).dt.hour == 21].copy()

            nbm_tc = (station_df['NBM Temperature (F)'] - 32) * (5.0 / 9.0)
            nbm_tdc = (station_df['NBM Dewpoint (F)'] - 32) * (5.0 / 9.0)
            nbm_es = np.exp((17.625 * nbm_tc) / (243.04 + nbm_tc))
            nbm_e = np.exp((17.625 * nbm_tdc) / (243.04 + nbm_tdc))
            station_df['NBM RH (%)'] = np.clip(100 * (nbm_e / nbm_es), 0.0, 100.0)

            # --- F. ELIMINATE UNNECESSARY COLUMNS ---
            columns_to_drop = [
                'NBM Initialization Time (UTC)',
                'NBM Forecast Valid (UTC)',
                'ASOS Dewpoint (F)',
                'NBM Dewpoint (F)',
                'init_time',
                'day_offset'
            ]
            raw_sounding_thermals = [c for c in station_df.columns if c.startswith('t_') or c.startswith('dpt_')]
            columns_to_drop.extend(raw_sounding_thermals)

            station_df = station_df.drop(columns=columns_to_drop, errors='ignore')
            station_df = station_df.dropna()

            print(f"      ✅ Generated {len(station_df)} valid tactical samples for {c_name}.")

            # Append strictly to this cycle's list
            cycle_master_dfs[c_name].append(station_df)

        except Exception as e:
            print(f"      ❌ Failed to process cycle {c_name} for K{station}: {e}")
            continue


# --- 3. EXPORT 4 INDIVIDUAL DATASETS ---

# Define your strict column hierarchy
TARGET_COLUMN_ORDER = [
    'Target Error (F)',
    'NBM Temperature (F)',
    'NBM Cloud Cover (%)',
    'NBM Mixing Height (100s ft AGL)',
    'NBM Wind Speed (kts)',
    'NBM Wind Direction (deg)',
    'NBM RH (%)',
    'hrrr_lpw (mm)',
    '1000mb-700mb Lapse Rate (C/km)',
    '700mb-500mb Lapse Rate (C/km)',
    'rh_1000', 'rh_975', 'rh_950', 'rh_925', 'rh_900', 'rh_875', 'rh_850',
    'rh_825', 'rh_800', 'rh_775', 'rh_750', 'rh_725', 'rh_700', 'rh_675',
    'rh_650', 'rh_625', 'rh_600', 'rh_575', 'rh_550', 'rh_525', 'rh_500',
    'sin_season',
    'cos_season'
]

print("\n🔗 Generating Individual Training Datasets...")

for cycle_name, df_list in cycle_master_dfs.items():
    if not df_list:
        print(f"   ⚠️ No data successfully compiled for {cycle_name}.")
        continue

    master_train_df = pd.concat(df_list, ignore_index=True)

    # Add shared engineering features
    day_of_year_series = master_train_df['valid_time'].dt.dayofyear
    master_train_df['sin_season'] = np.sin(2 * np.pi * day_of_year_series / 365.25)
    master_train_df['cos_season'] = np.cos(2 * np.pi * day_of_year_series / 365.25)

    # Set index and ensure datetime format
    master_train_df = master_train_df.set_index('valid_time')
    master_train_df.index = pd.to_datetime(master_train_df.index)

    # 📅 APPLY DATE THRESHOLD FILTER
    original_len = len(master_train_df)
    master_train_df = master_train_df.sort_index().loc[START_DATE:END_DATE]
    print(f"   📅 Applied date filter ({START_DATE} to {END_DATE}): {original_len} -> {len(master_train_df)} samples remaining.")

    # 🚀 ENFORCE STRICT COLUMN ORDER
    # Filter to only columns that actually exist in the DataFrame to prevent KeyErrors
    existing_cols = [col for col in TARGET_COLUMN_ORDER if col in master_train_df.columns]
    master_train_df = master_train_df[existing_cols]

    output_filename = f"TdAI_Training_Data_{cycle_name}.csv"
    output_dir = os.path.join(base_path, 'Training_Dataset')
    os.makedirs(output_dir, exist_ok=True)
    output_full_path = os.path.join(output_dir, output_filename)

    master_train_df.to_csv(output_full_path, index=True)
    print(f"   💾 Saved {cycle_name} Dataset: {len(master_train_df)} total samples -> {output_filename}")

print("\n✨ ALL DONE! Matrix configurations successfully generated.")

🚀 Starting Master Compilation for 1 stations across 4 distinct models...

────────────────── Processing Station: KCAR ──────────────────
   Loading ASOS ground truth...

   ⚙️ Structuring Cycle: 12z_f09...
      ✅ Generated 1555 valid tactical samples for 12z_f09.

   ⚙️ Structuring Cycle: 12z_f33...
      ✅ Generated 1545 valid tactical samples for 12z_f33.

   ⚙️ Structuring Cycle: 00z_f21...
      ✅ Generated 1536 valid tactical samples for 00z_f21.

   ⚙️ Structuring Cycle: 00z_f45...
      ✅ Generated 1372 valid tactical samples for 00z_f45.

🔗 Generating Individual Training Datasets...
   📅 Applied date filter (2020-12-02 to 2026-12-31): 1555 -> 1383 samples remaining.
   💾 Saved 12z_f09 Dataset: 1383 total samples -> TdAI_Training_Data_12z_f09.csv
   📅 Applied date filter (2020-12-02 to 2026-12-31): 1545 -> 1372 samples remaining.
   💾 Saved 12z_f33 Dataset: 1372 total samples -> TdAI_Training_Data_12z_f33.csv
   📅 Applied date filter (2020-12-02 to 2026-12-31): 1536 -> 1372 sam

# **Analyze the Td Bust Distribution**

In [4]:
# =========================================================================
# 📊 ANALYZE EXTREME DEWPOINT BUST DISTRIBUTION (PER OPERATIONAL CYCLE)
# =========================================================================

print("\n" + "="*75)
print("📊 EXTREME NBM DEWPOINT BUST ANALYSIS (≥ 5°F ABSOLUTE ERROR)")
print("="*75)

# Track global counts across all cycles
global_samples = 0
global_moist_count = 0
global_dry_count = 0

# Loop through each individual cycle stored in cycle_master_dfs
for cycle_name, df_list in cycle_master_dfs.items():
    if not df_list:
        print(f"\n⚠️ Cycle {cycle_name}: No samples compiled.")
        continue

    cycle_df = pd.concat(df_list, ignore_index=True)
    total_samples = len(cycle_df)

    if total_samples > 0:
        # Calculate positive errors (NBM significantly moister than ASOS)
        moist_busts = cycle_df[cycle_df['Target Error (F)'] >= 5.0]
        # Calculate negative errors (NBM significantly drier than ASOS)
        dry_busts = cycle_df[cycle_df['Target Error (F)'] <= -5.0]
        # Total absolute errors >= 5°F
        total_busts = cycle_df[cycle_df['Target Error (F)'].abs() >= 5.0]

        pct_moist = (len(moist_busts) / total_samples) * 100
        pct_dry = (len(dry_busts) / total_samples) * 100
        pct_total = (len(total_busts) / total_samples) * 100

        # Accumulate for global summary
        global_samples += total_samples
        global_moist_count += len(moist_busts)
        global_dry_count += len(dry_busts)

        print(f"\n⚙️ Cycle: {cycle_name} (Total Samples: {total_samples})")
        print(f"   🔹 Moist Busts (NBM Too Wet ≥ +5°F): {len(moist_busts):4d} samples ({pct_moist:5.2f}%)")
        print(f"   🔸 Dry Busts   (NBM Too Dry ≤ -5°F): {len(dry_busts):4d} samples ({pct_dry:5.2f}%)")
        print(f"   🔺 Total Extreme Busts (|Error| ≥ 5°F): {len(total_busts):4d} samples ({pct_total:5.2f}%)")
        print("──────────────────────────────────────────────────────────────────")

# -------------------------------------------------------------------------
# GLOBAL ALL-CYCLE SUMMARY
# -------------------------------------------------------------------------
if global_samples > 0:
    global_total_busts = global_moist_count + global_dry_count
    g_pct_moist = (global_moist_count / global_samples) * 100
    g_pct_dry = (global_dry_count / global_samples) * 100
    g_pct_total = (global_total_busts / global_samples) * 100

    print("\n🌐 OVERALL COMBINED DATASET SUMMARY (ALL CYCLES)")
    print(f"   🔹 Global Moist Busts (NBM Too Wet): {global_moist_count:4d} / {global_samples} ({g_pct_moist:5.2f}%)")
    print(f"   🔸 Global Dry Busts   (NBM Too Dry): {global_dry_count:4d} / {global_samples} ({g_pct_dry:5.2f}%)")
    print(f"   🔺 Global Total Extreme Busts      : {global_total_busts:4d} / {global_samples} ({g_pct_total:5.2f}%)")
    print("="*75)
else:
    print("\n❌ No valid samples found across any cycle to perform bust analysis.")


📊 EXTREME NBM DEWPOINT BUST ANALYSIS (≥ 5°F ABSOLUTE ERROR)

⚙️ Cycle: 12z_f09 (Total Samples: 1555)
   🔹 Moist Busts (NBM Too Wet ≥ +5°F):  105 samples ( 6.75%)
   🔸 Dry Busts   (NBM Too Dry ≤ -5°F):   60 samples ( 3.86%)
   🔺 Total Extreme Busts (|Error| ≥ 5°F):  165 samples (10.61%)
──────────────────────────────────────────────────────────────────

⚙️ Cycle: 12z_f33 (Total Samples: 1545)
   🔹 Moist Busts (NBM Too Wet ≥ +5°F):  133 samples ( 8.61%)
   🔸 Dry Busts   (NBM Too Dry ≤ -5°F):   92 samples ( 5.95%)
   🔺 Total Extreme Busts (|Error| ≥ 5°F):  225 samples (14.56%)
──────────────────────────────────────────────────────────────────

⚙️ Cycle: 00z_f21 (Total Samples: 1536)
   🔹 Moist Busts (NBM Too Wet ≥ +5°F):  109 samples ( 7.10%)
   🔸 Dry Busts   (NBM Too Dry ≤ -5°F):   74 samples ( 4.82%)
   🔺 Total Extreme Busts (|Error| ≥ 5°F):  183 samples (11.91%)
──────────────────────────────────────────────────────────────────

⚙️ Cycle: 00z_f45 (Total Samples: 1372)
   🔹 Moist Busts

# **Train the TdAI Probabilistic Models**

In [5]:
# =========================================================================
# 🎛️ CONFIGURATION SWITCHES
# =========================================================================
# True  -> Trains on ALL data (2021-2026) for optimal live deployment.
# False -> Leaves out 2025 strictly as an independent validation test bench.
PRODUCTION_MODE = True

base_path = "/content/drive/MyDrive/Fire Weather Focal Point/TdAI_Project/Development/Data/"

# Define the cycles to iterate through
CYCLES = ['00z_f21', '00z_f45', '12z_f09', '12z_f33']
target_quantiles = [0.10, 0.25, 0.50, 0.75, 0.90]

for cycle in CYCLES:
    print(f"\n=========================================================================")
    print(f"🔄 INITIATING PROBABILISTIC TRAINING FOR CYCLE: {cycle}")
    print(f"=========================================================================")

    # --- 0. LOAD THE SPECIFIC CYCLE DATASET ---
    data_path = os.path.join(base_path, f"Training_Dataset/TdAI_Training_Data_{cycle}.csv")
    if not os.path.exists(data_path):
        print(f"⚠️ Missing training data for {cycle}. Skipping...")
        continue

    master_train_df = pd.read_csv(data_path)

    # --- 1. PREPARE THE INDEX ---
    if 'valid_time' in master_train_df.columns:
        master_train_df = master_train_df.set_index('valid_time')

    # Guarantee the index is a parsed DatetimeIndex for perfect year slices
    master_train_df.index = pd.to_datetime(master_train_df.index)

    # --- 2. PREPARE X AND Y ---
    X = master_train_df.drop(columns=['Target Error (F)'], errors='ignore')
    y = master_train_df['Target Error (F)']

    # --- 3. DATA SPLIT SEGMENTATION ---
    if PRODUCTION_MODE:
        print("🚀 PRODUCTION MODE ACTIVATED: Training ensemble on 100% of available historical dataset...")
        X_train = X.copy()
        y_train = y.copy()
        X_test = pd.DataFrame()
        y_test = pd.Series(dtype=float)
        print(f"   └── Total Unified Operational Training Matrix: {len(X_train)} samples")
    else:
        print("🛡️ DEVELOPMENT VALIDATION MODE: Isolating 2025 convective archive...")
        years = X.index.year
        is_test_year = (years == 2025)
        is_train_year = ~is_test_year

        X_train = X[is_train_year]
        y_train = y[is_train_year]
        X_test = X[is_test_year]
        y_test = y[is_test_year]

        print(f"   ├── 🏋️ Training set size (2021-2024 + 2026): {len(X_train)} samples")
        print(f"   └── 🛡️ Test/Validation set size (Strictly 2025): {len(X_test)} samples")

    # Define sample weights (weight the moist bust days higher)
    sample_weights = np.where(y_train >= 5.0, 5.0,
                     np.where(y_train >= 3.0, 2.0, 1.0))

    probabilistic_models = {}

    print("🏋️ Initializing Probabilistic Quantile Regression Training...")

    for q in target_quantiles:
        print(f"   ├── Training GBDT Core Weights for Quantile: {q*100:.0f}th Percentile...")

        active_lambda = 8.0 if q == 0.50 else 2.0

        # Initialize the regressor using Quantile Loss
        model = LGBMRegressor(
            objective='regression_l1' if q == 0.50 else 'quantile', # LightGBM prefers L1 for 50th, quantile for others
            alpha=q,                  # This tells the model which exact percentile to lock onto
            n_estimators=80,          # Number of trees
            learning_rate=0.04,       # How much weight given to each new tree (generally want it small)
            min_child_samples=30,     # Before a prediction rule is created N samples must match that rule
            max_depth=4,              # How complex the trees can get. Low values force the model to focus on big, broad rules
            reg_lambda=active_lambda, # Penalizes the model for giving massive, outsized prediction weights to any single leaf choice.
            random_state=42,          # Ensures model reproducibility
            colsample_bytree=0.65,    # The percentage of predictors from which the model can choose its splits.
            verbose=-1                # Suppress warnings
        )

        # Train the model normally using your identical feature schema
        model.fit(X_train, y_train, sample_weight=sample_weights)

        # Store the trained weights inside a container dictionary matrix specific to this cycle
        probabilistic_models[f"q{int(q*100)}"] = model

    # =============================================================================================
    # 💾 SAVE PIPELINE: Export the Unified Probabilistic Model Dictionary (only in production mode)
    # =============================================================================================
    # 1. Isolate the exact list of string column names used for training
    trained_features = list(X_train.columns)

    # 2. Define your destination file paths mapping dynamically to the cycle
    trained_models_dir = os.path.join(base_path, "Trained_TdAI_Models/")
    os.makedirs(trained_models_dir, exist_ok=True)

    model_path = os.path.join(trained_models_dir, f"tdai_probabilistic_model_{cycle}.joblib")
    schema_path = os.path.join(trained_models_dir, f"probabilistic_model_feature_schema_{cycle}.joblib")

    print("💾 Exporting operational artifacts to disk...")

    try:
        # Save the 5-model group bundle for this specific cycle
        joblib.dump(probabilistic_models, model_path, compress=3)
        print(f"   ├── ✅ Probabilistic Ensemble exported to: '{os.path.basename(model_path)}'")

        # Save the column ordering list
        joblib.dump(trained_features, schema_path)
        print(f"   └── ✅ Feature Schema list exported to:      '{os.path.basename(schema_path)}'")
        print(f"       📊 Live script will enforce tracking for all {len(trained_features)} features.")

    except Exception as e:
        print(f"   ❌ Error exporting operational artifacts for {cycle}: {e}")

print("\n✨ ALL CYCLES COMPLETE! All probabilistic models trained and exported.")



🔄 INITIATING PROBABILISTIC TRAINING FOR CYCLE: 00z_f21
🚀 PRODUCTION MODE ACTIVATED: Training ensemble on 100% of available historical dataset...
   └── Total Unified Operational Training Matrix: 1372 samples
🏋️ Initializing Probabilistic Quantile Regression Training...
   ├── Training GBDT Core Weights for Quantile: 10th Percentile...
   ├── Training GBDT Core Weights for Quantile: 25th Percentile...
   ├── Training GBDT Core Weights for Quantile: 50th Percentile...
   ├── Training GBDT Core Weights for Quantile: 75th Percentile...
   ├── Training GBDT Core Weights for Quantile: 90th Percentile...
💾 Exporting operational artifacts to disk...
   ├── ✅ Probabilistic Ensemble exported to: 'tdai_probabilistic_model_00z_f21.joblib'
   └── ✅ Feature Schema list exported to:      'probabilistic_model_feature_schema_00z_f21.joblib'
       📊 Live script will enforce tracking for all 32 features.

🔄 INITIATING PROBABILISTIC TRAINING FOR CYCLE: 00z_f45
🚀 PRODUCTION MODE ACTIVATED: Training ensem

# **Validate the TdAI Probabilistic Models on the 2025 Test Set**

In [6]:
# =============================================================================================
# 🔍 VALIDATION PIPELINE: Load Joblib Models & Run Spectrum Predictions
# =============================================================================================
base_path = "/content/drive/MyDrive/Fire Weather Focal Point/TdAI_Project/Development/Data/"
CYCLES = ['00z_f21', '00z_f45', '12z_f09', '12z_f33']

for cycle in CYCLES:
    if not PRODUCTION_MODE:
        print(f"\n==================================================")
        print(f"🔍 RUNNING 2025 VALIDATION FOR {cycle}...")
        print(f"==================================================")

        # --- 1. LOAD THE SPECIFIC CYCLE DATASET & ISOLATE 2025 ---
        data_path = os.path.join(base_path, f"Training_Dataset/TdAI_Training_Data_{cycle}.csv")
        if not os.path.exists(data_path):
            print(f"   ⚠️ Missing data for {cycle}. Skipping...")
            continue

        cycle_df = pd.read_csv(data_path)
        cycle_df['valid_time'] = pd.to_datetime(cycle_df['valid_time'])
        cycle_df = cycle_df.set_index('valid_time')

        # Isolate strictly 2025
        is_test_year = (cycle_df.index.year == 2025)
        test_df = cycle_df[is_test_year]

        if len(test_df) == 0:
            print(f"   ⚠️ No 2025 test data found for {cycle}. Skipping...")
            continue

        X_test = test_df.drop(columns=['Target Error (F)'], errors='ignore')
        y_test = test_df['Target Error (F)']

        # --- 2. LOAD THE SPECIFIC TRAINED MODEL FOR THIS CYCLE ---
        model_path = os.path.join(base_path, f"Trained_TdAI_Models/tdai_probabilistic_model_{cycle}.joblib")
        if not os.path.exists(model_path):
            print(f"   ⚠️ Model file not found at {model_path}. Skipping...")
            continue

        loaded_probabilistic_models = joblib.load(model_path)

        # --- 3. GENERATE SPECTRUM PREDICTIONS ON THE 2025 TEST SET ---
        validation_results = pd.DataFrame(index=X_test.index)
        validation_results['True_Error'] = y_test

        # Extract predictions from each sub-model in the loaded dictionary
        for q_key, sub_model in loaded_probabilistic_models.items():
            validation_results[q_key] = sub_model.predict(X_test)

        # --- 4. CALCULATE POINT ACCURACY METRICS (MEDIAN VS RAW NBM) ---
        # Raw NBM Error is when the model predicts 0 correction bias
        nbm_mae = np.mean(np.abs(validation_results['True_Error']))
        tdai_median_mae = np.mean(np.abs(validation_results['True_Error'] - validation_results['q50']))

        print(f"📊 POINT FORECAST ACCURACY COMPILATION:")
        print(f"   ├── Baseline Raw NBM Mean Absolute Error: {nbm_mae:.2f}°F")
        print(f"   ├── TdAI Probabilistic Median (q50) MAE:  {tdai_median_mae:.2f}°F")
        skill_improvement = ((nbm_mae - tdai_median_mae) / nbm_mae) * 100
        print(f"   └── TdAI Error Reduction Skill Score:     {skill_improvement:.1f}%")

        # --- 5. CALIBRATE THE PROBABILISTIC SPREAD GAP (80% SPREAD COVERAGE) ---
        # Check how often the true error falls between your 10th and 90th percentile boundaries
        is_inside_80_band = (validation_results['True_Error'] >= validation_results['q10']) & \
                            (validation_results['True_Error'] <= validation_results['q90'])

        actual_coverage = np.mean(is_inside_80_band) * 100

        print(f"\n🎯 PROBABILISTIC ENSEMBLE CALIBRATION TARGET:")
        print(f"   ├── Theoretical Target Coverage Window: 80.0%")
        print(f"   ├── Actual Observed 2025 Data Coverage: {actual_coverage:.1f}%")

        if abs(actual_coverage - 80.0) <= 5.0:
            print("   └── Status: ✅ High Calibration Quality. The prediction interval reliably bounds operational uncertainty.")
        else:
            print("   └── Status: ⚠️ Calibration Skewed. The spread may be too wide or too narrow for extreme edges.")

In [7]:
# =========================================================================
# 🧮 TOP 25 BUST ANALYTICS & PLOTTING PIPELINE FOR ALL CYCLES
# =========================================================================
base_path = "/content/drive/MyDrive/Fire Weather Focal Point/TdAI_Project/Development/Data/"
CYCLES = ['00z_f21', '00z_f45', '12z_f09', '12z_f33']

# Map cycle codes to clean plot title labels
CYCLE_LABELS = {
    '00z_f21': '00z_Day 1',
    '00z_f45': '00z_Day 2',
    '12z_f09': '12z Day 1',
    '12z_f33': '12z Day 2'
}

for cycle in CYCLES:
    if not PRODUCTION_MODE:
        print(f"\n==================================================")
        print(f"📊 RUNNING TOP 25 BUST ANALYSIS FOR {cycle}...")
        print(f"==================================================")
        cycle_label = CYCLE_LABELS.get(cycle, cycle)

        # --- 1. LOAD THE SPECIFIC CYCLE DATASET & ISOLATE 2025 ---
        data_path = os.path.join(base_path, f"Training_Dataset/TdAI_Training_Data_{cycle}.csv")
        if not os.path.exists(data_path):
            print(f"   ⚠️ Missing data file for {cycle}. Skipping...")
            continue

        cycle_df = pd.read_csv(data_path)
        cycle_df['valid_time'] = pd.to_datetime(cycle_df['valid_time'])
        cycle_df = cycle_df.set_index('valid_time')

        # Isolate strictly 2025 test data
        is_test_year = (cycle_df.index.year == 2025)
        test_df = cycle_df[is_test_year]

        if len(test_df) == 0:
            print(f"   ⚠️ No 2025 test data found for {cycle}. Skipping...")
            continue

        X_test = test_df.drop(columns=['Target Error (F)'], errors='ignore')
        y_test = test_df['Target Error (F)']

        # --- 2. LOAD BOTH DETERMINISTIC AND PROBABILISTIC MODELS ---
        det_model_path = os.path.join(base_path, f"Trained_TdAI_Models/tdai_deterministic_model_{cycle}.joblib")
        prob_model_path = os.path.join(base_path, f"Trained_TdAI_Models/tdai_probabilistic_model_{cycle}.joblib")

        if not os.path.exists(det_model_path) or not os.path.exists(prob_model_path):
            print(f"   ⚠️ Missing trained model file(s) for {cycle}. Skipping...")
            continue

        gb_model = joblib.load(det_model_path)
        loaded_probabilistic_models = joblib.load(prob_model_path)

        # --- 3. GENERATE PREDICTIONS & BUILD RESULTS FRAMEWORK ---
        validation_results = pd.DataFrame(index=X_test.index)
        validation_results['True_Error'] = y_test

        # Extract probabilistic quantile predictions
        for q_key, sub_model in loaded_probabilistic_models.items():
            validation_results[q_key] = sub_model.predict(X_test)

        # Extract deterministic model prediction
        validation_results['Deterministic_TdAI'] = gb_model.predict(X_test)

        # --- 4. ISOLATE TOP 25 NBM BUST DAYS & CALCULATE SUBSET CALIBRATION ---
        bust_days_df = validation_results.sort_values(by='True_Error', ascending=False).head(25)
        plot_df = bust_days_df.sort_index()

        is_inside_50_band = (plot_df['True_Error'] >= plot_df['q25']) & (plot_df['True_Error'] <= plot_df['q75'])
        is_inside_80_band = (plot_df['True_Error'] >= plot_df['q10']) & (plot_df['True_Error'] <= plot_df['q90'])

        coverage_50 = is_inside_50_band.mean() * 100
        coverage_80 = is_inside_80_band.mean() * 100

        print(f"🎯 HIGH-IMPACT BUST SUBSET CALIBRATION ({cycle}):")
        print(f"   ├── 50% Interval Target Coverage Window: 50.0%  |  Actual Observed: {coverage_50:.1f}%")
        print(f"   └── 80% Interval Target Coverage Window: 80.0%  |  Actual Observed: {coverage_80:.1f}%")

        # --- 5. GENERATE THE HIGH-IMPACT DEVIATION CHART ---
        plt.style.use('default')
        plt.figure(figsize=(15, 7), dpi=100)
        ax = plt.gca()
        ax.set_facecolor('#f8fafc')

        x_labels = plot_df.index.strftime('%Y-%m-%d')
        x_indices = np.arange(len(plot_df))

        # 80% Boundary Envelope (q10 to q90)
        plt.vlines(
            x_indices, plot_df['q10'], plot_df['q90'],
            colors='#6366f1', alpha=0.25, linewidth=10,
            label='80% Probabilistic Confidence Range (q10 - q90)'
        )

        # 50% Boundary Envelope (q25 to q75)
        plt.vlines(
            x_indices, plot_df['q25'], plot_df['q75'],
            colors='#6366f1', alpha=0.55, linewidth=10,
            label='50% Probabilistic Confidence Range (q25 - q75)'
        )

        # TdAI Probabilistic Median Forecast Corrections (q50)
        plt.scatter(
            x_indices, plot_df['q50'],
            color='#4338ca', marker='_', s=250, linewidths=3, zorder=4,
            label='TdAI Probabilistic Median Correction (q50)'
        )

        # Deterministic TdAI Predictions Overlay
        plt.scatter(
            x_indices, plot_df['Deterministic_TdAI'],
            color='#f59e0b', marker='_', s=250, linewidths=3, zorder=4,
            label='Deterministic TdAI Prediction'
        )

        # Actual Ground Truth Errors recorded by ASOS
        plt.scatter(
            x_indices, plot_df['True_Error'],
            color='#e11d48', edgecolors='#ffffff', linewidths=1.0, s=80, zorder=5,
            label='Actual Observed NBM Error'
        )

        # --- CHART DECORATIONS & LIGHT STYLING ---
        plt.axhline(0, color='#64748b', alpha=0.6, linestyle=':', linewidth=1.5)

        plt.title(f"TdAI Probabilistic Verification [{cycle_label} Forecast]: Top 25 NBM Bust Days\n"
                  f"Reports in 25th-75th Band (50% confidence interval) = {coverage_50:.1f}% | Reports in 10th-90th Band (80% confidence interval) = {coverage_80:.1f}%",
                  fontsize=13, fontweight='bold', pad=14, color='#0f172a')

        plt.ylabel("Predicted NBM Td Error (°F)", fontdict={'weight': 'bold', 'color': '#0f172a'})
        plt.xlabel("NBM Bust Dates (Chronological Sequence)", fontdict={'weight': 'bold', 'color': '#0f172a'})

        plt.xticks(x_indices, x_labels, rotation=45, ha='right')
        plt.grid(True, color='#e2e8f0', linestyle='-', linewidth=0.5, axis='y')

        ax.tick_params(colors='#334155', labelsize=10)

        plt.legend(
            loc='upper left',
            frameon=True,
            facecolor='#ffffff',
            edgecolor='#cbd5e1',
            labelcolor='#1e293b',
            fontsize=9
        )

        plt.tight_layout()
        plt.show()

In [8]:
# =========================================================================
# 📈 CONTINUOUS TIME-SERIES CHRONOLOGICAL PLOTTING PIPELINE FOR ALL CYCLES
# =========================================================================
base_path = "/content/drive/MyDrive/Fire Weather Focal Point/TdAI_Project/Development/Data/"
CYCLES = ['00z_f21', '00z_f45', '12z_f09', '12z_f33']

for cycle in CYCLES:
    if not PRODUCTION_MODE:
        print(f"\n==================================================")
        print(f"📈 GENERATING CONTINUOUS TIME-SERIES PLOT FOR {cycle}...")
        print(f"==================================================")

        # --- 1. LOAD THE SPECIFIC CYCLE DATASET & ISOLATE 2025 ---
        data_path = os.path.join(base_path, f"Training_Dataset/TdAI_Training_Data_{cycle}.csv")
        if not os.path.exists(data_path):
            print(f"   ⚠️ Missing data file for {cycle}. Skipping...")
            continue

        cycle_df = pd.read_csv(data_path)
        cycle_df['valid_time'] = pd.to_datetime(cycle_df['valid_time'])
        cycle_df = cycle_df.set_index('valid_time')

        # Isolate strictly 2025 test data
        is_test_year = (cycle_df.index.year == 2025)
        test_df = cycle_df[is_test_year]

        if len(test_df) == 0:
            print(f"   ⚠️ No 2025 test data found for {cycle}. Skipping...")
            continue

        X_test = test_df.drop(columns=['Target Error (F)'], errors='ignore')
        y_test = test_df['Target Error (F)']

        # --- 2. LOAD PROBABILISTIC MODEL FOR THIS CYCLE ---
        prob_model_path = os.path.join(base_path, f"Trained_TdAI_Models/tdai_probabilistic_model_{cycle}.joblib")
        if not os.path.exists(prob_model_path):
            print(f"   ⚠️ Missing probabilistic model file for {cycle}. Skipping...")
            continue

        loaded_probabilistic_models = joblib.load(prob_model_path)

        # --- 3. GENERATE PREDICTIONS & BUILD RESULTS FRAMEWORK ---
        validation_results = pd.DataFrame(index=X_test.index)
        validation_results['True_Error'] = y_test

        # Extract probabilistic quantile predictions
        for q_key, sub_model in loaded_probabilistic_models.items():
            validation_results[q_key] = sub_model.predict(X_test)

        # Sort by index time axis for a clean chronological layout plot & slice 80 samples
        plot_df = validation_results.sort_index().head(80)

        # --- 4. GENERATE THE TIME-SERIES PLOT ---
        plt.style.use('default')
        plt.figure(figsize=(14, 6), dpi=100)
        ax = plt.gca()
        ax.set_facecolor('#f8fafc')

        # High uncertainty band (q10 to q90)
        plt.fill_between(
            plot_df.index,
            plot_df['q10'],
            plot_df['q90'],
            color='#6366f1',
            alpha=0.12,
            label='80% Probabilistic Confidence Band (q10 - q90)'
        )

        # Moderate uncertainty band (q25 to q75)
        plt.fill_between(
            plot_df.index,
            plot_df['q25'],
            plot_df['q75'],
            color='#6366f1',
            alpha=0.22,
            label='50% Probabilistic Confidence Band (q25 - q75)'
        )

        # Core median forecast trendline
        plt.plot(
            plot_df.index,
            plot_df['q50'],
            color='#4338ca',
            linewidth=2,
            linestyle='--',
            label='TdAI Median Forecast Correction (q50)'
        )

        # Actual ground-truth observation scatter points
        plt.scatter(
            plot_df.index,
            plot_df['True_Error'],
            color='#e11d48',
            edgecolors='#ffffff',
            linewidth=0.8,
            s=55,
            zorder=5,
            label='Actual Observed NBM Error'
        )

        # --- CHART DECORATIONS & LIGHT STYLING ---
        plt.axhline(0, color='#64748b', alpha=0.6, linestyle=':', linewidth=1.5)

        plt.title(f"TdAI Probabilistic Quantile Verification Analysis [{cycle}] (2025 - FIrst 80 days)",
                  fontsize=13, fontweight='bold', pad=12, color='#0f172a')
        plt.ylabel("Baseline Forecast Deviation Error Scale (°F)", fontdict={'weight': 'bold', 'color': '#0f172a'})
        plt.xlabel("Validation Timeline (Chronological)", fontdict={'weight': 'bold', 'color': '#0f172a'})

        plt.grid(True, color='#e2e8f0', linestyle='-', linewidth=0.8)
        ax.tick_params(colors='#334155', labelsize=10)

        plt.legend(
            loc='upper left',
            frameon=True,
            facecolor='#ffffff',
            edgecolor='#cbd5e1',
            labelcolor='#1e293b'
        )

        plt.tight_layout()
        plt.show()